# COOX — Outskirt Booking Analysis & Geo-Blocking Strategy

**Problem:** COOX receives bookings from locations outside serviceable areas (typically city outskirts or beyond operational limits). Since these bookings cannot be fulfilled, they are eventually cancelled and refunded — leading to revenue loss and poor customer experience.

**Objective:** Analyze recent historical booking data to identify geographical clusters where such non-serviceable bookings are most frequent, extract corresponding pin codes, and proactively block these areas to prevent future bookings.

---

| # | Deliverable | Description |
|---|-------------|-------------|
| 1 | Data Analysis | Distribution of outskirt bookings, city-wise trends, problem areas |
| 2 | Geospatial Mapping | Scatter maps, heat maps, cluster maps |
| 3 | Cluster Identification | DBSCAN-based density clustering with parameter justification |
| 4 | Pincode Extraction | Reverse geocoding of cluster centroids |
| 5 | Geo-Blocking Recommendation | Prioritized list of pin codes to block |

---
## 1. Setup & Data Loading

In [ ]:
!pip install geopy -q

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

In [ ]:
df = pd.read_csv('IIT Roorkee __ COOX - Raw Data.csv')
print(f'Dataset Shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
df.head()

---
## 2. Data Cleaning & Preprocessing

In [ ]:
# Standardize column names
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')
print('Cleaned columns:', list(df.columns))

In [ ]:
# Missing values check
print('Missing Values:')
print(df.isnull().sum())
print(f'\nRows with missing coordinates: {df[["lat","long"]].isnull().any(axis=1).sum()}')

In [ ]:
# Convert coordinate columns and drop invalid rows
df['lat'] = pd.to_numeric(df['lat'], errors='coerce')
df['long'] = pd.to_numeric(df['long'], errors='coerce')

rows_before = len(df)
df = df.dropna(subset=['lat', 'long'])
print(f'Dropped {rows_before - len(df)} rows with missing coordinates')
print(f'Working dataset: {len(df)} bookings')

In [ ]:
# Duplicate check
print(f'Duplicate rows: {df.duplicated().sum()}')
print(f'Duplicate Booking IDs: {df["booking_id"].duplicated().sum()}')

---
## 3. Exploratory Data Analysis

Key question: Where are these non-serviceable bookings concentrated — by city, area, and address type?

In [ ]:
# Confirm: all bookings are refunded
print('Payment Status Distribution:')
print(df['payment_status'].value_counts())
print(f'\nConfirmed: 100% of bookings are "{df["payment_status"].unique()[0]}" — entire dataset represents non-serviceable outskirt orders.')

In [ ]:
# City-wise outskirt booking breakdown
city_counts = df['city'].value_counts()
city_pct = df['city'].value_counts(normalize=True) * 100

city_summary = pd.DataFrame({
    'Booking Count': city_counts,
    'Percentage (%)': city_pct.round(2)
})
print('City-wise Outskirt Bookings:')
city_summary

In [ ]:
# City-wise distribution — bar and pie charts
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

colors = sns.color_palette('Reds_r', n_colors=len(city_counts))
city_counts.plot(kind='barh', ax=axes[0], color=colors, edgecolor='black')
axes[0].set_title('Outskirt Bookings by City', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Number of Refunded Bookings')
axes[0].invert_yaxis()

top_cities = city_counts.head(6)
other = city_counts.iloc[6:].sum()
pie_data = pd.concat([top_cities, pd.Series({'Others': other})])
pie_data.plot(kind='pie', ax=axes[1], autopct='%1.1f%%', startangle=90,
              colors=sns.color_palette('Set2', len(pie_data)))
axes[1].set_title('Share of Outskirt Bookings', fontsize=14, fontweight='bold')
axes[1].set_ylabel('')

plt.tight_layout()
plt.show()

In [ ]:
# State-wise distribution
print('State-wise Distribution:')
print(df['state'].value_counts())

In [ ]:
# Address type breakdown
addr_type = df['address_type'].value_counts()
addr_pct = df['address_type'].value_counts(normalize=True) * 100

print('Address Type Distribution:')
pd.DataFrame({'Count': addr_type, 'Percentage (%)': addr_pct.round(2)})

In [ ]:
# Cross-tabulation: City x Address Type
ct = pd.crosstab(df['city'], df['address_type'])

plt.figure(figsize=(10, 8))
sns.heatmap(ct, annot=True, fmt='d', cmap='YlOrRd', linewidths=0.5)
plt.title('City x Address Type Heatmap (Outskirt Bookings)', fontsize=14, fontweight='bold')
plt.xlabel('Address Type')
plt.ylabel('City')
plt.tight_layout()
plt.show()

In [ ]:
# Top 20 problem areas by booking count
top_areas = df['area'].value_counts().head(20)
print('Top 20 Problem Areas:')
print(top_areas)

top10_count = df['area'].value_counts().head(10).sum()
print(f'\nTop 10 areas account for {top10_count}/{len(df)} = {(top10_count/len(df)*100):.1f}% of all outskirt bookings')

In [ ]:
# Repeat offender addresses — same address generating multiple failed bookings
repeat_addresses = df['address_id'].value_counts()
repeat_addresses = repeat_addresses[repeat_addresses > 1]

print(f'Addresses with multiple outskirt bookings: {len(repeat_addresses)}')
print(f'Total bookings from repeat addresses: {repeat_addresses.sum()}')
print(f'\nTop 10 repeat addresses:')

for addr_id in repeat_addresses.head(10).index:
    addr_data = df[df['address_id'] == addr_id].iloc[0]
    print(f'  Address ID {addr_id}: {repeat_addresses[addr_id]} bookings -- {addr_data["area"]}, {addr_data["city"]}')

### EDA Key Findings

1. **100% of the dataset is refunded bookings** — confirming these are all non-serviceable outskirt orders
2. **Bengaluru dominates** with the highest number of outskirt bookings, followed by Pune and Hyderabad
3. **Home is the primary address type** — customers ordering to residential addresses in outskirts
4. **Repeat offenders exist** — same addresses generating multiple failed bookings
5. **Top 10 problem areas account for a significant share** of all outskirt bookings

---
## 4. Geospatial Visualization

In [ ]:
import folium
from folium.plugins import HeatMap

In [ ]:
# Scatter map of all outskirt booking locations
m = folium.Map(
    location=[df['lat'].mean(), df['long'].mean()],
    zoom_start=5,
    tiles='CartoDB positron'
)

for _, row in df.iterrows():
    folium.CircleMarker(
        location=[row['lat'], row['long']],
        radius=3,
        color='crimson',
        fill=True,
        fill_opacity=0.5,
        popup=f"{row['area']}, {row['city']}<br>Type: {row['address_type']}<br>Booking: {row['booking_id']}"
    ).add_to(m)

m

In [ ]:
# Heatmap — booking density visualization
m_heat = folium.Map(
    location=[df['lat'].mean(), df['long'].mean()],
    zoom_start=5,
    tiles='CartoDB dark_matter'
)

HeatMap(
    df[['lat', 'long']].values.tolist(),
    radius=15,
    blur=10,
    max_zoom=13,
    gradient={0.2: 'blue', 0.4: 'lime', 0.6: 'yellow', 0.8: 'orange', 1: 'red'}
).add_to(m_heat)

m_heat

---
## 5. Cluster Identification (DBSCAN)

**Why DBSCAN?**
- Does not require pre-specifying the number of clusters (unlike K-Means)
- Can find arbitrarily shaped clusters (outskirt zones are not circular)
- Automatically identifies noise/outlier points
- Works natively with haversine distance for geographic coordinates

In [ ]:
from sklearn.cluster import DBSCAN
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import silhouette_score

In [ ]:
# k-Distance plot for optimal eps selection
coords = df[['lat', 'long']].to_numpy()
kms_per_radian = 6371.0088

k = 5
neighbors = NearestNeighbors(n_neighbors=k, metric='haversine')
neighbors.fit(np.radians(coords))
distances, _ = neighbors.kneighbors(np.radians(coords))
k_distances = np.sort(distances[:, -1]) * kms_per_radian

plt.figure(figsize=(12, 5))
plt.plot(k_distances, color='darkred', linewidth=2)
plt.xlabel('Points (sorted by distance)', fontsize=12)
plt.ylabel(f'{k}-th Nearest Neighbor Distance (km)', fontsize=12)
plt.title('k-Distance Graph for DBSCAN eps Selection', fontsize=14, fontweight='bold')
plt.axhline(y=2, color='blue', linestyle='--', linewidth=1.5, label='eps = 2 km')
plt.legend(fontsize=12)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f'Selected eps = 2 km (based on elbow in k-distance plot)')
print(f'Selected min_samples = {k} (standard minimum for geographic clustering)')

In [ ]:
# Run DBSCAN
eps_km = 2.0
epsilon = eps_km / kms_per_radian

db = DBSCAN(eps=epsilon, min_samples=k, metric='haversine')
df['cluster'] = db.fit_predict(np.radians(coords))

n_clusters = len(set(df['cluster'])) - (1 if -1 in df['cluster'].values else 0)
n_noise = (df['cluster'] == -1).sum()

print(f'Clusters found: {n_clusters}')
print(f'Noise points (isolated bookings): {n_noise}')
print(f'Clustered bookings: {len(df) - n_noise} / {len(df)} ({(len(df)-n_noise)/len(df)*100:.1f}%)')

In [ ]:
# Cluster quality — Silhouette Score
df_clustered = df[df['cluster'] != -1]

if len(df_clustered['cluster'].unique()) > 1:
    sil_score = silhouette_score(
        np.radians(df_clustered[['lat', 'long']]),
        df_clustered['cluster'],
        metric='haversine'
    )
    print(f'Silhouette Score: {sil_score:.3f}')
    if sil_score > 0.5:
        print('Interpretation: Good cluster separation')
    elif sil_score > 0.25:
        print('Interpretation: Moderate cluster separation')
    else:
        print('Interpretation: Weak separation — clusters may overlap')
else:
    print('Only 1 cluster found — silhouette score not applicable')

In [ ]:
# Cluster scatter plot
plt.figure(figsize=(14, 8))

noise = df[df['cluster'] == -1]
plt.scatter(noise['long'], noise['lat'], c='lightgray', s=20, alpha=0.5, label='Noise (isolated)')

clustered = df[df['cluster'] != -1]
scatter = plt.scatter(
    clustered['long'], clustered['lat'],
    c=clustered['cluster'], cmap='tab10',
    s=40, edgecolors='black', linewidths=0.5, alpha=0.8
)
plt.colorbar(scatter, label='Cluster ID')
plt.xlabel('Longitude', fontsize=12)
plt.ylabel('Latitude', fontsize=12)
plt.title('DBSCAN Clusters — Outskirt Booking Hotspots', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Cluster summary table
cluster_summary = df_clustered.groupby('cluster').agg(
    booking_count=('booking_id', 'count'),
    mean_lat=('lat', 'mean'),
    mean_long=('long', 'mean'),
    cities=('city', lambda x: ', '.join(x.unique())),
    top_area=('area', lambda x: x.value_counts().index[0]),
    address_types=('address_type', lambda x: ', '.join(x.unique()))
).sort_values('booking_count', ascending=False)

print('Cluster Summary:')
cluster_summary

In [ ]:
# Tag as Outskirt Hotspots
hotspot_threshold = 5
hotspot_clusters = cluster_summary[cluster_summary['booking_count'] >= hotspot_threshold]
minor_clusters = cluster_summary[cluster_summary['booking_count'] < hotspot_threshold]

print(f'OUTSKIRT HOTSPOTS (>={hotspot_threshold} bookings): {len(hotspot_clusters)} clusters')
print(hotspot_clusters[['booking_count', 'cities', 'top_area']])
print(f'\nMinor clusters (<{hotspot_threshold} bookings): {len(minor_clusters)} clusters')

---
## 6. Pincode Extraction

**Method:** Reverse geocoding of cluster centroids using Nominatim (OpenStreetMap). This provides the pin code, locality, and district for each identified outskirt hotspot.

*Note: Reverse geocoding requires API calls. This section takes approximately 2-5 minutes to run due to rate limiting.*

In [ ]:
# Method 1: Extract pincodes from address field using regex (6-digit Indian pincodes)
df['pincode_from_address'] = df['area'].astype(str).str.extract(r'(\d{6})')
pincodes_found = df['pincode_from_address'].notna().sum()
print(f'Pincodes found in address text: {pincodes_found} / {len(df)}')

if pincodes_found > 0:
    print('\nPincodes extracted from address field:')
    print(df[df['pincode_from_address'].notna()][['area', 'city', 'pincode_from_address']])

In [ ]:
# Method 2: Reverse geocoding for cluster centroids
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter

geolocator = Nominatim(user_agent='coox_outskirt_analysis_v1', timeout=10)
reverse = RateLimiter(geolocator.reverse, min_delay_seconds=1.5)

pincode_results = []

print('Reverse geocoding cluster centroids...\n')
for cluster_id, row in cluster_summary.iterrows():
    try:
        location = reverse(f"{row['mean_lat']}, {row['mean_long']}", language='en')
        if location and location.raw.get('address'):
            address = location.raw['address']
            pincode = address.get('postcode', 'N/A')
            suburb = address.get('suburb', address.get('village', address.get('town', 'N/A')))
            district = address.get('state_district', address.get('county', 'N/A'))
            state = address.get('state', 'N/A')
        else:
            pincode, suburb, district, state = 'N/A', 'N/A', 'N/A', 'N/A'
    except Exception as e:
        pincode, suburb, district, state = 'Error', 'Error', 'Error', str(e)

    pincode_results.append({
        'cluster': cluster_id,
        'pincode': pincode,
        'locality': suburb,
        'district': district,
        'state': state,
        'lat': row['mean_lat'],
        'long': row['mean_long'],
        'booking_count': row['booking_count'],
        'top_area': row['top_area'],
        'cities_affected': row['cities']
    })
    print(f'  Cluster {cluster_id}: Pincode {pincode} -- {suburb}, {district}')

pincode_df = pd.DataFrame(pincode_results)
print('\nPincode Extraction Results:')
pincode_df

In [ ]:
# Detailed pincodes for individual locations within hotspot clusters
top_cluster_ids = hotspot_clusters.index.tolist()
top_cluster_bookings = df[df['cluster'].isin(top_cluster_ids)]

unique_coords = top_cluster_bookings.drop_duplicates(subset=['lat', 'long'])[['lat', 'long', 'cluster', 'area', 'city']]
print(f'Reverse geocoding {len(unique_coords)} unique locations in hotspot clusters...')

individual_pincodes = []
for _, row in unique_coords.iterrows():
    try:
        location = reverse(f"{row['lat']}, {row['long']}", language='en')
        if location and location.raw.get('address'):
            pincode = location.raw['address'].get('postcode', 'N/A')
        else:
            pincode = 'N/A'
    except:
        pincode = 'Error'

    individual_pincodes.append({
        'cluster': row['cluster'],
        'area': row['area'],
        'city': row['city'],
        'pincode': pincode,
        'lat': row['lat'],
        'long': row['long']
    })

individual_pincode_df = pd.DataFrame(individual_pincodes)
print('\nIndividual Location Pincodes (Hotspot Clusters):')
individual_pincode_df

---
## 7. Geo-Blocking Recommendation

**Final Deliverable:** A prioritized list of pincodes and areas that COOX should block to prevent future non-serviceable bookings.

In [ ]:
# Geo-Blocking Recommendation Table
blocking_recommendation = pincode_df.copy()

def assign_priority(count):
    if count >= 10:
        return 'HIGH -- BLOCK IMMEDIATELY'
    elif count >= 5:
        return 'MEDIUM -- BLOCK RECOMMENDED'
    else:
        return 'LOW -- MONITOR'

blocking_recommendation['priority'] = blocking_recommendation['booking_count'].apply(assign_priority)
blocking_recommendation = blocking_recommendation.sort_values('booking_count', ascending=False)

print('=' * 80)
print('GEO-BLOCKING RECOMMENDATION')
print('=' * 80)
print()
blocking_recommendation[['cluster', 'pincode', 'locality', 'district', 'state',
                         'booking_count', 'cities_affected', 'priority']]

In [ ]:
# Impact analysis
total_blockable = blocking_recommendation[blocking_recommendation['booking_count'] >= 5]['booking_count'].sum()
total_all_clusters = blocking_recommendation['booking_count'].sum()

print('Impact Analysis:')
print(f'  Total bookings in identified clusters: {total_all_clusters}')
print(f'  Bookings preventable by blocking HIGH+MEDIUM priority areas: {total_blockable}')
print(f'  Prevention rate: {total_blockable/len(df)*100:.1f}% of all outskirt bookings')

In [ ]:
# Final map: Blocking zones visualization
m_final = folium.Map(
    location=[df['lat'].mean(), df['long'].mean()],
    zoom_start=5,
    tiles='CartoDB positron'
)

# All booking points
for _, row in df.iterrows():
    folium.CircleMarker(
        location=[row['lat'], row['long']],
        radius=2,
        color='gray',
        fill=True,
        fill_opacity=0.3
    ).add_to(m_final)

# Blocking zones
for _, row in blocking_recommendation.iterrows():
    if row['booking_count'] >= 10:
        zone_color = 'red'
    elif row['booking_count'] >= 5:
        zone_color = 'orange'
    else:
        zone_color = 'green'

    folium.Circle(
        location=[row['lat'], row['long']],
        radius=3000,
        color=zone_color,
        fill=True,
        fill_opacity=0.3,
        popup=folium.Popup(
            f"<b>Cluster {row['cluster']}</b><br>"
            f"Pincode: {row['pincode']}<br>"
            f"Locality: {row['locality']}<br>"
            f"Bookings: {row['booking_count']}<br>"
            f"Priority: {row['priority']}",
            max_width=300
        )
    ).add_to(m_final)

    folium.Marker(
        location=[row['lat'], row['long']],
        popup=f"Cluster {row['cluster']} | Pin: {row['pincode']} | {row['booking_count']} bookings",
        icon=folium.Icon(
            color='red' if row['booking_count'] >= 10 else 'orange',
            icon='ban-circle',
            prefix='glyphicon'
        )
    ).add_to(m_final)

# Legend
legend_html = '''
<div style="position: fixed; bottom: 50px; left: 50px; z-index: 1000; background-color: white;
     padding: 15px; border-radius: 5px; border: 2px solid gray; font-size: 14px;">
     <b>Geo-Blocking Zones</b><br>
     <i style="background:red; width:12px; height:12px; display:inline-block; border-radius:50%;"></i> HIGH -- Block Immediately (10+ bookings)<br>
     <i style="background:orange; width:12px; height:12px; display:inline-block; border-radius:50%;"></i> MEDIUM -- Block Recommended (5+ bookings)<br>
     <i style="background:green; width:12px; height:12px; display:inline-block; border-radius:50%;"></i> LOW -- Monitor (<5 bookings)<br>
     <i style="background:gray; width:12px; height:12px; display:inline-block; border-radius:50%;"></i> Individual outskirt bookings
</div>
'''
m_final.get_root().html.add_child(folium.Element(legend_html))

m_final

---
## 8. Limitations & Future Work

### Limitations
- **No temporal data:** The dataset lacks timestamps, so we cannot identify trends or seasonality in outskirt bookings
- **Single payment status:** All records are refunded — no comparison against successful deliveries is possible from this dataset alone
- **Sample size:** 557 records may not capture all problematic zones across India
- **No revenue data:** Cannot quantify the exact financial impact of each non-serviceable booking
- **Static analysis:** City boundaries evolve over time; outskirt zones today may become serviceable areas tomorrow

### Recommended Future Work
1. **Integrate successful delivery data** to build a serviceable vs. non-serviceable binary classifier
2. **Add temporal analysis** to track whether non-serviceable zones are expanding (city growth)
3. **Build a real-time geo-fence system** using the cluster boundaries identified here
4. **A/B test blocking strategies** — soft-block (warning to customer) vs. hard-block (prevent booking)
5. **Customer impact analysis** — track whether blocked customers rebook from serviceable addresses
6. **Dynamic re-clustering** — periodically re-run the pipeline as new booking data comes in

---
## 9. Executive Summary

### Key Findings
1. All 557 bookings in the dataset are fully refunded, confirming they represent non-serviceable outskirt orders
2. Bengaluru is the primary problem city, driven by farmhouses and residential addresses in peripheral villages
3. DBSCAN clustering identified distinct geographical hotspots across Bengaluru, Pune, Hyderabad, Chennai, Ahmedabad, and NCR
4. Pin codes were extracted via reverse geocoding for each cluster centroid
5. Blocking HIGH and MEDIUM priority zones can prevent a significant percentage of future non-serviceable bookings

### Recommendation
COOX should implement a tiered geo-fence blocking system:
- **HIGH priority zones** (10+ historical failed bookings): Block immediately — prevent booking placement
- **MEDIUM priority zones** (5+ bookings): Show a service availability warning to the customer at checkout
- **LOW priority zones** (under 5 bookings): Monitor and re-evaluate on a quarterly basis
- Allow manual override for edge cases with customer service approval